# LexChain RAG generation benchmark — justify the LLM choice

**Single-independent-variable experiment.** The pipeline is frozen to the chunk/embed winners — LangChain `RecursiveCharacterTextSplitter` (512/50) + `e5-base-v2`, reusing that benchmark's **cached embeddings** — and the *only* thing varied is the generation LLM.

**Models (Groq, durable post-Llama set):** `openai/gpt-oss-120b`, `openai/gpt-oss-20b`, `qwen/qwen3.6-27b`. (Groq's Llama 3.1/3.3 are EOL Aug 16 2026; Llama 3.1 70B already removed — so we benchmark what LexChain can actually deploy going forward.)

**Metric:** OHR-Bench's own generation scoring, vendored verbatim (`ohr_gen_eval.py`) — token-overlap **F1** (headline) + **EM** accuracy. Answers are produced with OHR-Bench's exact QA prompt (`<response>…</response>`, brief answers, "Not answerable").

**Resumable:** every answered question is checkpointed to Drive; a Groq daily rate-limit stops cleanly and the same command resumes the next day.

Runtime: CPU is fine (retrieval reuses cached embeddings; the LLM runs on Groq). A GPU runtime is only needed if the e5 embedding cache is absent.

In [ ]:
# Cell 1 — mount Drive, clone/pull, install, set GROQ_API_KEY
from google.colab import drive, userdata
drive.mount('/content/drive')

import os
if not os.path.exists('/content/lexchain-chunk-embed-bench'):
    !git clone https://github.com/MarvinPescos/lexchain-chunk-embed-bench.git /content/lexchain-chunk-embed-bench
%cd /content/lexchain-chunk-embed-bench
!git pull
!pip install -q -r requirements.txt

# Store your key once via the Colab 🔑 Secrets panel as GROQ_API_KEY (notebook access ON)
os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
print('GROQ_API_KEY set:', bool(os.environ.get('GROQ_API_KEY')))

In [ ]:
# Cell 2 — data + embedding cache. Reuses the chunk/embed benchmark's cached
# e5 embeddings on Drive (MyDrive/lexchain_bench/); embeds only if absent (needs GPU).
!python download_data.py
import os
emb = '/content/drive/MyDrive/lexchain_bench/emb/langchain__e5-base-v2.npz'
print('cached e5 chunk embeddings present:', os.path.exists(emb))
print('(if False, run the chunk/embed benchmark first, or this run will embed on GPU)')

In [ ]:
# Cell 3 — SMOKE: 8 stratified questions x 3 models, then project the full run
!python rag_generate.py --smoke 8
!python gen_estimate.py --targets 1142,200
!python gen_aggregate.py --suffix _smoke8

**STOP — review the projection above and choose the run size.**

The 70B-class models are capped at ~100–200K tokens/day on the free tier (~35–70 answers/day). The estimate cell prints paid cost and free-tier calendar days for both the **full 1,142** and a **stratified 200** sample. Recommended default: **`--sample 200`** (stratified by evidence source), which is statistically adequate and finishes in a few resumable days on free tier — or add a few $ of Groq credit for the full run in ~2 h. Edit the next cell accordingly.

In [ ]:
# Cell 4 — approved run (resumable). Rerun this same cell to resume after a
# disconnect or a daily rate-limit; answered questions are skipped.
# Full run instead: drop --sample 200.
!python rag_generate.py --sample 200

In [ ]:
# Cell 5 — results table (winner = end-to-end system) + human-review CSV
!python gen_aggregate.py --suffix _n200

import pandas as pd
from IPython.display import display, Markdown
cache = '/content/drive/MyDrive/lexchain_bench'
display(pd.read_csv(f'{cache}/gen_matrix_n200.csv'))
display(Markdown(open(f'{cache}/gen_results_table_n200.md').read()))
print('Human-review CSV for your team:', f'{cache}/human_review_n200.csv')
print('After they fill human_correct: '
      '!python gen_aggregate.py --with-human <that csv>  -> auto-vs-human agreement + kappa')